# ToysAD-8K Official Seen and Unseen Test Evaluation

**Notebook file:** `10b-toysad8k-official-seen-unseen-evaluation.ipynb`

This notebook opens the official seen-test and unseen-test evaluators only after both prediction locks and all frozen hashes verify. It reports the two test splits separately and adds a pooled descriptive table.

The final route is LocateAnything, SAM 2.1, DINOv2-small with P1 scoring,
label-free cross-view identity matching, mean normalized-rank fusion, and the
review policy frozen in Notebook 09B. The evaluator cannot alter any part of
that route.

## Evaluation contract

- Every prediction artifact must match its immutable prediction lock.
- The exact Notebook 08F method hash and Notebook 09B policy hash must match.
- The content-addressed execution contract must identify the exact cohort membership,
  prediction, run-record, and evaluator-package hashes.
- The completed Notebook 09D protocol receipt and the Notebook 10A summary must link the same method, policy, and both official prediction locks.
- Official prediction objects match evaluator references one-to-one at box IoU 0.50. Lower overlaps remain unmatched and cannot earn a correct odd-one-out result.
- Evaluator labels are parsed only after all non-label checks pass.
- A reused nonempty result directory is rejected.
- Failed stages, zero proposals, missing results, and review outcomes remain in
  the denominator.
- Results are reported for 2, 3, and 5 uploaded views.
- Base-asset-clustered confidence intervals preserve repeated-scene dependence.
- The notebook does not tune a threshold or apply a universal accuracy gate.
- ClearML starts only after the scientific outputs and hashes are complete.

Seen-test and unseen-test are the primary results. The pooled row is descriptive only. No official result may change the method or policy.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image as NotebookImage, display

sys.dont_write_bytecode = True

NOTEBOOK_FILE = "10b-toysad8k-official-seen-unseen-evaluation.ipynb"
STAGE = "10b"
CONTRACT_FILE = "evaluation_contract_10b.json"
EXECUTION_ENVIRONMENT = "kaggle" if Path("/kaggle/input").is_dir() else "local"
OUTPUT_ROOT = Path(os.getenv(
    "VPI_OUTPUT_ROOT",
    "/kaggle/working/toysad8k_official_seen_unseen_evaluation"
    if EXECUTION_ENVIRONMENT == "kaggle"
    else ".local/runs/toysad8k_official_seen_unseen_evaluation",
)).expanduser().resolve()

In [ ]:
def mounted_roots() -> list[Path]:
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        return sorted((path.resolve() for path in input_root.iterdir() if path.is_dir()), key=str)
    return [Path.cwd().resolve()]


def find_code_root() -> Path:
    configured = os.getenv("VPI_CODE_ROOT")
    candidates = [Path(configured).expanduser()] if configured else []
    candidates.extend((Path.cwd(), *Path.cwd().parents, *mounted_roots()))
    for candidate in candidates:
        if (candidate / "scripts/run_one_shot_evaluation.py").is_file():
            return candidate.resolve()
        if (candidate / "code/scripts/run_one_shot_evaluation.py").is_file():
            return (candidate / "code").resolve()
    raise FileNotFoundError("Could not locate the Visual Peer Inspector code snapshot")


def read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, keep_default_na=False, na_values=[""])


def display_images(paths: list[Path], limit: int | None = None) -> None:
    selected = paths if limit is None else paths[:limit]
    for path in selected:
        print(path.name)
        display(NotebookImage(filename=str(path)))


def run_tracking_finalizer(result_root: Path) -> dict:
    command = [
        sys.executable,
        str(CODE_ROOT / "scripts/run_one_shot_evaluation.py"),
        "track",
        "--result-root", str(result_root),
    ]
    receipt_path = result_root / "clearml_receipt.json"
    try:
        completed = subprocess.run(
            command,
            check=True,
            capture_output=True,
            text=True,
            timeout=240,
            env=os.environ.copy(),
        )
        print(completed.stdout.strip())
    except Exception as error:
        receipt = {
            "schema_version": "visual-peer-clearml-finalizer/v1",
            "status": "failed",
            "error_type": type(error).__name__,
            "scientific_output_preserved": True,
            "credentials_persisted": False,
        }
        temporary = receipt_path.with_suffix(".json.incomplete")
        temporary.write_text(json.dumps(receipt, indent=2, sort_keys=True) + "\n", encoding="utf-8")
        temporary.replace(receipt_path)
    return json.loads(receipt_path.read_text(encoding="utf-8"))

## Resolve and lock the inputs

The notebook first verifies the frozen method, review policy, prediction locks,
prediction-only run records, fixed cohort identities, and evaluator manifests.
It then writes their exact hashes to a local execution contract. Evaluator label
rows are not parsed during this step. A filename match alone is never accepted.

In [ ]:
CODE_ROOT = find_code_root()
sys.path.insert(0, str(CODE_ROOT / "src"))
from visual_peer_inspector.notebook_provenance import verify_notebook_code_snapshot

CODE_PROVENANCE = verify_notebook_code_snapshot(
    CODE_ROOT,
    "notebooks/official-evaluation/10b-toysad8k-official-seen-unseen-evaluation.ipynb",
)
MOUNTED_ROOTS = mounted_roots()
EVALUATION_CONTRACT = Path(
    "/kaggle/working" if EXECUTION_ENVIRONMENT == "kaggle" else ".local/runs"
) / CONTRACT_FILE

contract_command = [
    sys.executable,
    str(CODE_ROOT / "scripts/run_one_shot_evaluation.py"),
    "contract",
    "--stage", STAGE,
    "--output", str(EVALUATION_CONTRACT),
]
for root in MOUNTED_ROOTS:
    contract_command.extend(("--mount-root", str(root)))
subprocess.run(contract_command, check=True, timeout=1800)

from visual_peer_inspector.workflows.one_shot_evaluation import load_evaluation_contract

contract = load_evaluation_contract(EVALUATION_CONTRACT, expected_stage=STAGE)
display({
    "notebook": NOTEBOOK_FILE,
    "stage": STAGE,
    "contract SHA-256": contract.contract_sha256,
    "method SHA-256": contract.method_record_sha256,
    "policy SHA-256": contract.policy_record_sha256,
    "cohorts": [item.cohort_id for item in contract.cohorts],
    "expected splits": 'seen-test and unseen-test',
})

## Run the sealed evaluation

The command verifies all model-facing files first, writes a boundary receipt,
and only then opens the evaluator package. Output files are written through
temporary paths and renamed only when complete.

In [ ]:
if OUTPUT_ROOT.exists() and any(OUTPUT_ROOT.iterdir()):
    raise FileExistsError("The one-shot output directory is already in use")

command = [
    sys.executable,
    str(CODE_ROOT / "scripts/run_one_shot_evaluation.py"),
    "evaluate",
    "--stage", STAGE,
    "--evaluation-contract", str(EVALUATION_CONTRACT),
    "--output-root", str(OUTPUT_ROOT),
    "--bootstrap-repetitions", "10000",
    "--panel-count", "16",
]
for root in MOUNTED_ROOTS:
    command.extend(("--mount-root", str(root)))

subprocess.run(command, check=True, timeout=7200)
completion = json.loads((OUTPUT_ROOT / "evaluation_complete.json").read_text(encoding="utf-8"))
if completion.get("status") != "complete":
    raise RuntimeError("Evaluation did not produce a complete atomic result")
display(completion)

## Final odd-one-out results

Top-1 is the web application's primary single-candidate result. Average
precision retains the full ranking and includes missed anomalous objects in its
denominator. Recall at two measures recovery of up to two anomalous objects,
while exact-set accuracy checks whether the first `anomaly_count` identities are
exactly the complete anomalous set. Top-2 and MRR retain the first-target view of
the ranking. Coverage includes every scene, including pipeline failures.
Selective accuracy describes only results released without a review flag, while
selective risk is its error rate.

In [ ]:
SAFE_ROOT = OUTPUT_ROOT / "safe"
summary = read_csv(SAFE_ROOT / "summary_metrics.csv")
intervals = read_csv(SAFE_ROOT / "clustered_confidence_intervals.csv")
display(summary)
display(intervals)
display_images([
    SAFE_ROOT / "figures/ranking_by_view.png",
    SAFE_ROOT / "figures/coverage_and_review.png",
])

## Reliability and subgroup checks

Margin and agreement are diagnostic signals, not probabilities. Their fixed
bins show whether larger values correspond to better empirical results. The
subgroup tables expose category, anomaly-type, object-count, and view-count
variation without changing the frozen policy.

In [ ]:
reliability = read_csv(SAFE_ROOT / "margin_agreement_reliability.csv")
subgroups = read_csv(SAFE_ROOT / "subgroup_and_category_metrics.csv")
display(reliability)
display(subgroups)
display_images([SAFE_ROOT / "figures/margin_reliability.png"])

## Pipeline attrition, identity failures, and error taxonomy

The attrition table follows evaluator-supplied target boxes through proposals,
nonempty SAM masks, cross-view identity tracks, and ranked tracks. Box and mask
IoU are reported only when the locked evaluator supplies the corresponding
geometry. The official ToysAD evaluation therefore measures anomalous target
retention rather than full normal-object detection recall. The error table
separates missing outputs, identity errors, wrong automatic results, and
results correctly sent for review.

In [ ]:
for table_name in (
    "pipeline_attrition.csv",
    "identity_failures.csv",
    "prediction_failures.csv",
    "runtime_summary.csv",
    "decision_confusion.csv",
    "error_taxonomy.csv",
):
    print(table_name)
    display(read_csv(SAFE_ROOT / table_name))
display_images([SAFE_ROOT / "figures/pipeline_attrition.png"])

## Private qualitative evidence

Sixteen deterministic panels are selected across best, typical, wrong, and
failure outcomes where those outcomes exist. Each panel shows the input views,
LocateAnything boxes, SAM masks, P1 evidence, the fused odd-one-out, and private
reference overlays. These panels are evaluator-only and must not enter a
model-facing or public application package.

In [ ]:
inventory = json.loads((OUTPUT_ROOT / "review-only/qualitative_panel_inventory.json").read_text(encoding="utf-8"))
display(pd.DataFrame(inventory["items"]))
panel_paths = sorted((OUTPUT_ROOT / "review-only").rglob("panel-*.png"))
if len(panel_paths) != inventory["panel_count"]:
    raise RuntimeError("Panel files differ from the qualitative inventory")
display_images(panel_paths)

## Bounded ClearML finalization

ClearML receives aggregate CSV tables and charts only. Scene-level labels,
crosswalks, RGB files, masks, and private panels remain local to this evaluator
run. A tracking timeout or service error creates a sanitized failure receipt and
does not invalidate the completed evaluation.

In [ ]:
clearml_receipt = run_tracking_finalizer(OUTPUT_ROOT)
display(clearml_receipt)

## Final locked record

The tables above are measurements, not settings. The method and policy remain
unchanged after this notebook. Any later report must cite the result hashes and
must preserve separate failure, review, and insufficient-evidence outcomes.
Official evaluator outputs remain separated from all model-facing resources.

In [ ]:
display(summary.loc[summary['split_name'].isin(['seen-test', 'unseen-test', 'pooled-descriptive'])])